In [8]:
import sys
sys.path.append("..")

import json
import torch
import pandas as pd
import optuna

from src.data import create_dataloader
from src.models import build_mlp
from src.train import train_model



In [9]:
train_df = pd.read_csv("../data/processed/train.csv")

target_col = "class"
feature_cols = [c for c in train_df.columns if c != target_col]

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [10]:
from sklearn.model_selection import StratifiedKFold
from src.train import train_one_epoch, validate

def objective(trial):

    
    hidden_layers = trial.suggest_categorical(
        "hidden_layers",
        [
            [64],
            [128],
            [256],
            [64, 32],
            [128, 64],
            [256, 128],
            [128, 128, 64]
        ]
    )

    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    grad_clip = trial.suggest_float("grad_clip", 0.5, 5.0)

    # --- 5-fold CV ---
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    val_losses = []

    for train_idx, val_idx in skf.split(train_df, train_df[target_col]):
        train_fold = train_df.iloc[train_idx]
        val_fold   = train_df.iloc[val_idx]

        train_loader = create_dataloader(train_fold, feature_cols, target_col,
                                         batch_size=batch_size, shuffle=True)
        val_loader   = create_dataloader(val_fold,   feature_cols, target_col,
                                         batch_size=batch_size, shuffle=False)

        model = build_mlp(
            config={"hidden_layers": hidden_layers, "dropout": dropout},
            input_dim=len(feature_cols),
            output_dim=train_df[target_col].nunique()
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        best_val_loss = float("inf")
        best_state = None

        for epoch in range(10):  
            train_loss = train_one_epoch(model, train_loader, optimizer, device, grad_clip)
            val_loss, _ = validate(model, val_loader, device)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = model.state_dict()

        val_losses.append(best_val_loss)

    return sum(val_losses) / len(val_losses)


In [14]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, catch=(Exception,))

for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        print("Trial", t.number, "FAILED with error:")
        print(t.state, t.values, t.params)
        print(t.system_attrs.get("fail_reason"))
        print("-" * 80)

study.best_value, study.best_params


[I 2026-04-23 17:15:26,116] A new study created in memory with name: no-name-e937c96b-f572-44fb-bc25-f3818f311f15
/home/dw/Desktop/mlp-openml-project/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [64] which is of type list.
  optuna_warn(message)
/home/dw/Desktop/mlp-openml-project/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [128] which is of type list.
  optuna_warn(message)
/home/dw/Desktop/mlp-openml-project/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [256] which is of type list.
  optuna_warn(message)
/home/dw/De

(0.1910351361192408,
 {'hidden_layers': [256, 128],
  'dropout': 0.006637638206590968,
  'lr': 0.004019188571880907,
  'batch_size': 64,
  'grad_clip': 4.785360125621526})

In [18]:

sorted_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value
)

top3 = sorted_trials[:3]
top3


[FrozenTrial(number=13, state=<TrialState.COMPLETE: 1>, values=[0.1910351361192408], datetime_start=datetime.datetime(2026, 4, 23, 17, 24, 56, 838927), datetime_complete=datetime.datetime(2026, 4, 23, 17, 25, 32, 735381), params={'hidden_layers': [256, 128], 'dropout': 0.006637638206590968, 'lr': 0.004019188571880907, 'batch_size': 64, 'grad_clip': 4.785360125621526}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'hidden_layers': CategoricalDistribution(choices=([64], [128], [256], [64, 32], [128, 64], [256, 128], [128, 128, 64])), 'dropout': FloatDistribution(high=0.5, log=False, low=0.0, step=None), 'lr': FloatDistribution(high=0.01, log=True, low=0.0001, step=None), 'batch_size': CategoricalDistribution(choices=(32, 64, 128, 256)), 'grad_clip': FloatDistribution(high=5.0, log=False, low=0.5, step=None)}, trial_id=13, value=None),
 FrozenTrial(number=15, state=<TrialState.COMPLETE: 1>, values=[0.21952616597215333], datetime_start=datetime.datetime(2026, 4, 23

In [19]:
import os
import json

os.makedirs("ensemble_configs", exist_ok=True)

for i, t in enumerate(top3):
    with open(f"ensemble_configs/top_{i+1}.json", "w") as f:
        json.dump(t.params, f, indent=4)

print("Saved top‑3 configs.")


Saved top‑3 configs.
